# TennisMyLife — Colab ALTO + RapidOCR worker

Manual batch worker for Gallica pages. It reads a claimed manifest from the VPS, uses ALTO for `mode=ALTO`, RapidOCR for `mode=RAPID`, and writes compatible outputs directly into the selected VPS OCR cache.

Create these Colab Secrets: `VPS_HOST`, `VPS_USER`, `VPS_SSH_KEY_B64`, `VPS_PORT`, `VPS_MANIFEST`, `VPS_REMOTE_CACHE`, and optionally `VPS_ALTO_CACHE`.
Never put secrets in the notebook or repository.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!pip -q install -r /content/Tennis-OCR-Pipeline/colab/requirements.txt


In [ ]:
from google.colab import userdata
import base64, paramiko

VPS_HOST = userdata.get('VPS_HOST')
VPS_USER = userdata.get('VPS_USER')
VPS_KEY_B64 = userdata.get('VPS_SSH_KEY_B64')
VPS_PORT = int(userdata.get('VPS_PORT') or 22)
VPS_MANIFEST = userdata.get('VPS_MANIFEST')
VPS_REMOTE_CACHE = userdata.get('VPS_REMOTE_CACHE')
VPS_ALTO_CACHE = userdata.get('VPS_ALTO_CACHE') or ''
assert all([VPS_HOST,VPS_USER,VPS_KEY_B64,VPS_MANIFEST,VPS_REMOTE_CACHE]), 'Missing Colab Secrets'
print('Configuration loaded; secrets are not printed.')


In [ ]:
from io import StringIO
key_text = base64.b64decode(VPS_KEY_B64).decode()
key = None
for cls in (paramiko.Ed25519Key, paramiko.RSAKey, paramiko.ECDSAKey):
    try:
        key = cls.from_private_key(StringIO(key_text)); break
    except Exception:
        pass
assert key is not None, 'Unsupported SSH key format'
tr = paramiko.Transport((VPS_HOST, VPS_PORT)); tr.connect(username=VPS_USER, pkey=key)
sftp = paramiko.SFTPClient.from_transport(tr)
local_manifest = '/content/colab_claim.tsv'
sftp.get(VPS_MANIFEST, local_manifest)
sftp.close(); tr.close()
print('Manifest downloaded:', local_manifest)
!head -5 /content/colab_claim.tsv


In [ ]:
PROFILE = 'HQ'       # HQ or STANDARD
ALTO_FIRST = False   # keep False: manifest mode controls the split
MAX_PAGES = 0        # 0 = whole claim
DELAY = 15.0         # ALTO pacing

cmd = ['python','/content/Tennis-OCR-Pipeline/colab/worker.py',
       '--manifest','/content/colab_claim.tsv',
       '--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',VPS_KEY_B64,
       '--vps-port',str(VPS_PORT),'--remote-cache',VPS_REMOTE_CACHE,
       '--profile',PROFILE,'--delay',str(DELAY),'--max-pages',str(MAX_PAGES)]
if VPS_ALTO_CACHE: cmd += ['--remote-alto-cache', VPS_ALTO_CACHE]
if ALTO_FIRST: cmd.append('--alto-first')
import subprocess
subprocess.run(cmd, check=True)


## Resume behavior
Each finished page is uploaded immediately to the final VPS OCR cache. Re-running the same claim skips pages whose JSON already exists remotely, so an interrupted Colab session can resume safely.
